# evaluate memote of curated models and fix biomass precursors, deadends and orphans. 

In [1]:
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra import Reaction, Metabolite
import os
import pandas as pd
from cobra.util.solver import solvers
import re
import json

import sys
sys.path.insert(0, '/home/emma/Dokumente/thesis')

from functions import *

### Evaluate post lib MEMOTE reports

In [ ]:
report_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_mb2_lib_bz/MEMOTE_reports"
model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_mb2_lib_bz"
save_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/final"

In [3]:
def extract_window_data(html_file):
    """Extract window.data dictionary from MEMOTE HTML report"""
    with open(html_file, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    # Find window.data = {...}
    match = re.search(r'window\.data\s*=\s*({.*?});', html_content, re.DOTALL)
    if match:
        json_str = match.group(1)
        return json.loads(json_str)
    else:
        raise ValueError("Could not find window.data in HTML file")


In [ ]:
def summarize_reports(report_dir):
    blocked_reactions_dict = {}
    orphaned_mets_dict = {}
    deadend_mets_dict = {}
    connectivity_dict = {}
    mapping_dict = {}

    for file in os.listdir(report_dir):
        if file.endswith('.html'):
            report_key = file.replace('.html', '') 
            html_file = os.path.join(report_dir, file)
            try:
                data_dict = extract_window_data(html_file)
                
                # 1. Blocked Reactions
                blocked_reactions = data_dict["tests"].get("test_blocked_reactions", {}).get("data", [])
                for reac in blocked_reactions:
                    blocked_reactions_dict.setdefault(reac, []).append(report_key)
                
                # 2. Orphaned Metabolites
                orphaned_mets = data_dict["tests"].get("test_find_orphans", {}).get("data", [])
                for met in orphaned_mets:
                    orphaned_mets_dict.setdefault(met, []).append(report_key)
                    mapping_dict.setdefault(met, []).append(report_key)
                
                # 3. Dead-end Metabolites
                deadend_mets = data_dict["tests"].get("test_find_deadends", {}).get("data", [])
                for met in deadend_mets:
                    deadend_mets_dict.setdefault(met, []).append(report_key)
                    mapping_dict.setdefault(met, []).append(report_key)
                
                # 4. Metabolite Connectivity
                connectivity_data = data_dict["tests"].get("test_find_disconnected", {}).get("data", [])
                for met in connectivity_data:
                    connectivity_dict.setdefault(met, []).append(report_key)
                    mapping_dict.setdefault(met, []).append(report_key)

            except Exception as e:
                print(f"Error parsing structural data from {file}: {e}")
                
    return blocked_reactions_dict, orphaned_mets_dict, deadend_mets_dict, connectivity_dict, mapping_dict

In [30]:
b_rxns, o_mets, d_mets, c_mets, mapping_dict = summarize_reports(report_dir)

#### fixes

##### gram positive fix

In [19]:
def add_gp_cellwall(model):
    add_new_met(model, "uGgla_c", "UDP-N-acetylmuramoyl-L-alanyl-gamma-D-glutamyl-L-lysyl-D-alanyl-D-alanine", "C40H62N9O26P2", -3, "c")
    add_new_met(model, "uGgl_c", "UDP-N-acetylmuramoyl-L-alanyl-gamma-D-glutamyl-L-lysine", "C34H52N7O24P2", -3, "c")

    add_new_rxn(model, "UAAGLS2", "UDP-N-acetylmuramoyl-L-alanyl-D-glutamyl-L-lysine synthetase (gamma-glutamate)", 0, 1000, {"lys__L_c": -1.0, "atp_c": -1.0, "uamag_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "uGgl_c": 1.0, "h_c": 1.0})
    add_new_rxn(model, "UGLDDS2", "UDP-N-acetylmuramoyl-L-alanyl-D-glutamyl-L-lysyl-D-alanyl-D-alanine synthetase (gamma-glutamate)", 0, 1000, {"alaala_c": -1.0, "atp_c": -1.0, "uGgl_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "uGgla_c": 1.0})
    add_new_rxn(model, "PAPPT2", "Phospho-N-acetylmuramoyl-pentapeptide-transferase (gamma-glutamate)", 0, 1000, {"udcpp_c": -1.0, "uGgla_c": -1.0, "ump_c": 1.0, "uaGgla_c": 1.0})

In [23]:
def overwrite_formula(model, rxn_id, new_formula):
    if rxn_id in model.metabolites:
        met = model.metabolites.get_by_id(rxn_id)
        old_formula = met.formula
        met.formula = new_formula

In [24]:
def total_gp_specific(model):
    if model.id in gram_pos_models:
        overwrite_reaction(model, "PPTGS_BS", {"uaaGgla_c": -1.0, "h_c":1.0, "peptido_BS_c": 1.0, "udcpdp_c":1.0 })
        overwrite_formula(model, "peptido_BS_c", "C39H63N8O19")
        add_gp_cellwall(model)


##### mql8 fix

In [32]:
# manually add q8h2_c and other required metabolites
q8h2_c = Metabolite(
    'q8h2_c',
    formula='C49H76O4', name='Ubiquinol-8', compartment='c')

ohph_c = Metabolite(
    '2ohph_c',
    formula='C46H70O2', name='2-Octaprenyl-6-hydroxyphenol',compartment='c')

omph_c = Metabolite(
    '2omph_c',
    formula='C47H72O2', name='2-Octaprenyl-6-methoxyphenol',compartment='c')

ombzl_c = Metabolite(
    '2ombzl_c',
    formula='C47H72O3', name='2-Octaprenyl-6-methoxy-1,4-benzoquinol', compartment='c')

ommbl_c = Metabolite(
    '2ommbl_c',
    formula='C48H74O3', name='2-Octaprenyl-3-methyl-6-methoxy- 1,4-benzoquinol', compartment='c')

omhmbl_c = Metabolite(
    '2omhmbl_c',
    formula='C48H74O4', name='2-Octaprenyl-3-methyl-5-hydroxy-6-methoxy-1,4-benzoquinol', compartment='c')

In [33]:
def add_ubi_metabolites(model):    
    add_new_met(model, "4hbz_e", '4-Hydroxybenzoate', 'C7H5O3', -1, "C_e")
    add_new_met(model, "4hbz_p", '4-Hydroxybenzoate', 'C7H5O3', -1, "C_p")
    add_new_met(model, "4hbz_c", '4-Hydroxybenzoate', 'C7H5O3', -1, "C_c")
    add_new_met(model, "3ophb_c", '3-Octaprenyl-4-hydroxybenzoate', 'C47H69O3', -1, "C_c")
    add_new_met(model, "2oph_c", '2-Octaprenylphenol', 'C46H70O', 0, "C_c")
    add_new_met(model, "2ohph_c", '2-Octaprenyl-6-hydroxyphenol', 'C46H70O2', 0, "C_c") # Lisa only adds down from here
    add_new_met(model, "2omph_c",'2-Octaprenyl-6-methoxyphenol', 'C47H72O2', 0, "_c")
    add_new_met(model, "2ombzl_c",'2-Octaprenyl-6-methoxy-1,4-benzoquinol', 'C47H72O3', 0, "C_c")
    add_new_met(model, "2ommbl_c",'2-Octaprenyl-3-methyl-6-methoxy-1,4-benzoquinol', 'C48H74O3', 0, "C_c")
    add_new_met(model, "2omhmbl_c",'2-Octaprenyl-3-methyl-5-hydroxy-6-methoxy-1,4-benzoquinol', 'C48H74O4', 0, "C_c")
    add_new_met(model, "q8h2_c",'Ubiquinol-8', 'C49H76O4', 0, "C_c")


In [34]:
# list of reactions that need to be added
# EX_4hbz_e: 4hbz_e ⇌ 
# 4HBZtex: 4hbz_e ⇌ 4hbz_p
# 4HBZt3pp: 4hbz_c + h_p ⇌ h_c + 4hbz_p
# HBZOPT: 4hbz_c + octdp_c ⇌ 3ophb_c + ppi_c
# OPHBDC: 3ophb_c + h_c ⇌ 2oph_c + co2_c
# OPHHX: 2oph_c + 0.5 o2_c ⇌ 2ohph_c 
# 2.1.1.222 = OHPHM: 2ohph_c + amet_c → 2omph_c + ahcys_c + h_c #lisa adds downstream from here
# 1.14.13.- = OMPHHX: 2omph_c + 0.5 o2_c → 2ombzl_c
# 2.1.1.201 = OMBZLM: 2ombzl_c + amet_c → 2ommbl_c + ahcys_c + h_c
# 1.14.99.60 = OMMBLHX: 2ommbl_c + 0.5 o2_c → 2omhmbl_c
# 2.1.1.64 = DMQMT: 2omhmbl_c + amet_c → ahcys_c + h_c + q8h2_c


In [35]:
# taken from Lisa - # add synthesis reactions 2.1.1.222 (OHPHM), 1.14.13.- (OMPHHX), 2.1.1.201 (OMBLZM), 1.14.99.60 (OMMBLHX) and 2.1.1.64 (DMQMT), directionality as done by Lisa 
def add_EX_4hbz_e(model): #need to add EX_4hbz_e to medium!!!
    add_new_rxn(model, "EX_4hbz_e", "Exchange 4hbz", -1000, 1000,
                {"4hbz_e": 1.0})
    
def add_4HBZtex(model):
    add_new_rxn(model, "4HBZtex", "4-Hydroxybenzoate transport (extracellular)", 0, 1000,
                {"4hbz_e": -1.0, "4hbz_p": 1.0})
    
def add_4HBZt3pp(model):
    add_new_rxn(model, "4HBZt3pp", "4-hydroxybenzoate transport out via antiporter", 0, 1000,
                {"4hbz_p": -1.0, "h_p": -1.0, "4hbz_c": 1.0, "h_c": 1.0})
    
def add_HBZOPT(model):
    add_new_rxn(model, "HBZOPT", "Hydroxybenzoate octaprenyltransferase", 0, 1000,
                {"4hbz_c": -1.0, "octdp_c": -1.0, '3ophb_c': 1.0, "ppi_c": 1.0})
    
def add_OPHBDC(model):
    add_new_rxn(model, "OPHBDC", "Octaprenyl-hydroxybenzoate decarboxylase", 0, 1000,
                {"3ophb_c": -1.0, "h_c": -1.0, '2oph_c': 1.0, "co2_c": 1.0})
    
def add_OPHHX(model):
    add_new_rxn(model, "OPHHX", "2-Octaprenylphenol hydroxylase", 0, 1000,
            {'2oph_c': -1.0, "o2_c": -0.5, "2ohph_c": 1.0})
    
def add_OHPHM(model):
    add_new_rxn(model, "OHPHM", "2-octaprenyl-6-hydroxyphenol methylase", 0, 1000,
                {'2ohph_c': -1.0, '2omph_c': 1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0})

def add_OMPHHX(model):
    add_new_rxn(model, "OMPHHX", "2-octaprenyl-6-methoxyphenol hydroxylase", 0, 1000,
                {'2ombzl_c': 1.0, '2omph_c': -1.0, 'o2_c': -0.5})

def add_OMBZLM(model):
    add_new_rxn(model, "OMBZLM", "2-Octaprenyl-6-methoxy-benzoquinol methylase", 0, 1000,
                {'2ombzl_c': -1.0, '2ommbl_c': 1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0})

def add_OMMBLHX(model):
    add_new_rxn(model, "OMMBLHX", "2-Octaprenyl-3-methyl-6-methoxy-1,4-benzoquinol hydroxylase", 0, 1000,
                {'2omhmbl_c': 1.0, '2ommbl_c': -1.0, 'o2_c': -0.5})

def add_DMQMT(model):
    add_new_rxn(model, "DMQMT", "3-Dimethylubiquinonol 3-methyltransferase", 0, 1000,
                {'2omhmbl_c': -1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0, 'q8h2_c': 1.0})

In [36]:
def add_ubi_rxns(model):
    add_EX_4hbz_e(model)
    add_4HBZtex(model)
    add_4HBZt3pp(model)
    add_HBZOPT(model)
    add_OPHBDC(model)
    add_OPHHX(model)
    add_OHPHM(model)
    add_OMPHHX(model)
    add_OMBZLM(model)
    add_OMMBLHX(model)
    add_DMQMT(model)

In [37]:
def replace_metabolite_in_reaction(reaction, old_metabolite, new_metabolite):
    reaction_model = model.reactions.get_by_id(reaction)
    try:
        old_met_model = model.metabolites.get_by_id(old_metabolite)
    except KeyError:
        return
    try:
        new_met_model = model.metabolites.get_by_id(new_metabolite)
    except KeyError:
        new_met_model = new_metabolite
    if old_met_model in reaction_model.metabolites:
        coefficient = reaction_model.get_coefficient(old_met_model)
        reaction_model.add_metabolites({new_met_model: coefficient})
        reaction_model.add_metabolites({old_met_model: -coefficient})
    else:
        return

In [38]:
def total_ubi_flow(model):
    if model.id in ubi_models:

        add_ubi_metabolites(model)
        add_ubi_rxns(model)

        # add Ubiquinol to Biomass for 895 instead of Menaquinol
        growth_rxn = model.reactions.get_by_id("Growth")
        metm = model.metabolites.get_by_id("mql8_c")
        metq = model.metabolites.get_by_id("q8h2_c")
        if metm in growth_rxn.metabolites:
            growth_rxn.subtract_metabolites({metm: growth_rxn.metabolites[metm]})
        if metq not in growth_rxn.metabolites:
            growth_rxn.add_metabolites({metq: -0.0001})


#### Main

In [39]:
metabolites_to_delete = ["lgt_s_c", "fdxo_2_2_c", "2mb_p_c", "ptth_c","ptcys_c", "galctr__D_c"]

In [40]:
gram_pos_models = ["m_1334_", "m_2872_", "m_638_",  "m_978_", "m_1114_", "m_1208", "m_1350_", "m_2751_"] #exclude "m_709_", as this is a bacillus species that is able to use the meso stuff

In [41]:
ubi_models = ["m_895_"]

In [42]:
for file in os.listdir(model_dir):
    if not file.endswith(('.xml', '.sbml')):
        continue
        
    m_number = file.split('_')[0]
    report_key_name = "m_" + m_number + "_"
    print(report_key_name)

    if f'{file[:-4]}_fix.xml' not in os.listdir(save_dir):
        model = read_sbml_model(os.path.join(model_dir, file))
        
        # delete metabolites that are overwritten/otherwise obsolete
        for met in metabolites_to_delete:
            if met in model.metabolites:
                met_model = model.metabolites.get_by_id(met)
                reactions = list(met_model.reactions)
                model.remove_reactions(reactions)
                model.remove_metabolites(met_model)

        #add missing reactions only if the model is reported to have a gap there                
        if report_key_name in mapping_dict.get("5dh4dglc_c", []):
            add_new_rxn(model, "D4DGCD", "5 dehydro 4 deoxyglucarate dehydratase", 0, 1000, {"h_c": -1.0, "5d4dglcr_c": -1.0, "co2_c": 1.0, "h2o_c": 1.0, "25dop_c": 1.0})
            
        if report_key_name in mapping_dict.get("bglyg4n_c", []):
            add_new_rxn(model, "GLBRAN3", "1 4 alpha glucan branching enzyme glyg4n bglyg4n", 0, 1000, {"glyg4n_c": -1.0, "bglyg4n_c": 1.0})
            
        if report_key_name in mapping_dict.get("btn_c", []) or report_key_name in mapping_dict.get("dtbt_c", []) :
            add_new_rxn(model, "BTS_1", "Biotin synthase", 0, 1000, {"dtbt_c": -1.0, "s_c": -2.0, "btn_c": 1.0, "h_c": 1.0, "h2s_c": 1.0})
        
        if report_key_name in mapping_dict.get("acglu_c", []):
            add_new_met(model, "acg5p_c", "N-Acetyl-L-glutamyl 5-phosphate", "C7H9NO8P", -3, "c")
            add_new_rxn(model, "ACGK", "Acetylglutamate kinase", 0, 1000, {"acglu_c": -1.0, "atp_c": -1.0, "adp_c": 1.0, "acg5p_c": 1.0})

        if report_key_name in mapping_dict.get("din_c", []):
            add_new_rxn(model, "PUNP6_1", "Purine-nucleoside phosphorylase (Deoxyinosine)", -1000, 1000, {"din_c": -1.0, "h_c": -1.0, "pi_c": -1.0, "2dr1p_c": 1.0, "hxan_c": 1.0})

        if report_key_name in mapping_dict.get("phaccoa_c", []):
            add_new_rxn(model, "PACCOAL", "Phenylacetate-CoA ligase", -1000, 1000, {"coa_c": -1.0, "atp_c": -1.0, "pac_c": -1.0, "amp_c": 1.0, "ppi_c": 1.0, "phaccoa_c": 1.0})

        if report_key_name in mapping_dict.get("s_c", []):
            add_new_rxn(model, "EX_s_e", "Sulfur import", -1000, 1000, {"s_e": -1.0})
            add_new_rxn(model, "St", "Sulfur transport", -1000, 1000, {"s_e": -1.0, "s_c": 1.0})

        if report_key_name in mapping_dict.get("thm_c", []):
            add_new_rxn(model, "TMK", "Thiamine kinase", 0, 1000, {"atp_c": -1.0, "thm_c": -1.0, "adp_c": 1.0, "h_c":1.0, "thmmp_c": 1.0})

        total_ubi_flow(model)
        total_gp_specific(model)

        print(model.slim_optimize())

        write_sbml_model(model, os.path.join(save_dir, f'{file[:-4]}_fix.xml'))

m_790_
83.73419544274635
m_638_
54.17112312182419
m_761_
64.38903283581836
m_793_
67.65222467847445
m_504_
51.95734579016473
m_161_
63.68269227208385
m_1018_
63.37629448134004
m_644_
54.41029006489322
m_100_
54.53363608890081
m_1234_
83.42948956368647
m_778_
76.69525857384598
m_352_
54.97649332014465
m_1432_
79.76500136677335
m_1124_
53.46787178934061
m_997_
69.85160471881468
m_892_
51.732600670951165
m_947_
40.0559970020168
m_1391_
65.04306572102819
m_2862_
43.633827606569454
m_1174_
58.88615046167985
m_1357_
75.48943702309656
m_2751_
33.74394540823529
m_1208_
51.113557841401175
m_163_
80.46600623042539
m_2774_
51.3720601075707
m_1362_
68.57412960843085
m_1167_
43.741180356451785
m_1252_
50.19953581084187
m_895_
51.05032102748583
m_1101_
66.021351190538
m_230_
56.20210487432557
m_397_
63.008469550067915
m_939_
68.00889474732624
m_709_
83.10447836190441
m_1056_
50.91502176511579
m_262_
63.140612268267084
m_867_
57.40879469093115
m_796_
56.53162995356937
m_1080_
68.24216441762108
m_946_